<a href="https://colab.research.google.com/github/Jaychandrarevada/IN226019002_GEN_AI/blob/main/AI_Resume_Screening_System_with_Tracing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Resume Screening System with Tracing
This notebook implements a resume screening system that extracts text from documents and uses GenAI to evaluate candidates.

In [1]:
# CELL 1
!pip install -q langchain langchain-xai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 11.4 MB/s eta 0:00:00


In [14]:
# CELL 2
import os
from getpass import getpass

print("Enter your xAI (Grok) API Key:")
os.environ["XAI_API_KEY"] = getpass()

print("Enter your LangSmith API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass()

# Enable LangSmith Tracing (Mandatory requirement)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume_Screening_System_Grok"

Enter your xAI (Grok) API Key:
··········
Enter your LangSmith API Key:
··········


In [15]:
# CELL 3
!mkdir -p prompts chains

In [16]:
%%writefile prompts/screening_prompts.py
from langchain_core.prompts import PromptTemplate

# Prompt instructions enforcing constraints and preventing hallucinations
resume_eval_template = """
You are an expert AI Technical Recruiter. Your task is to evaluate a candidate's resume against a specific Job Description.

JOB DESCRIPTION:
{job_description}

CANDIDATE RESUME:
{resume_text}

INSTRUCTIONS:
1. Extract the candidate's skills, experience, and tools directly from the resume.
2. Compare the extracted data against the job description requirements.
3. Assign a fit score from 0 to 100 based on the match.
4. Provide a brief, logical explanation for the score.
5. STRICT RULE: Do NOT assume skills, tools, or experience not explicitly present in the resume. If a requirement is missing, note it as missing.

Format your response strictly as a JSON object with the following keys:
- "extracted_skills": (list of strings)
- "extracted_experience": (string summarizing years/roles)
- "extracted_tools": (list of strings)
- "fit_score": (integer between 0 and 100)
- "explanation": (string detailing why the score was given)
"""

# Using PromptTemplate as required
resume_eval_prompt = PromptTemplate(
    input_variables=["job_description", "resume_text"],
    template=resume_eval_template
)

Overwriting prompts/screening_prompts.py


In [17]:
# CELL 5
%%writefile chains/eval_chain.py
import os
from langchain_xai import ChatXAI
from langchain_core.output_parsers import JsonOutputParser
from prompts.screening_prompts import resume_eval_prompt

def get_screening_chain():
    # Grab the key explicitly from the environment
    api_key = os.getenv("XAI_API_KEY")

    # Initialize the Grok LLM with an active model string
    llm = ChatXAI(
        model="grok-3-mini",  # <-- Fixed: Using current xAI API model
        temperature=0,
        xai_api_key=api_key
    )

    # Initialize the JSON parser to ensure structured output
    parser = JsonOutputParser()

    # Build the LCEL Pipeline: Prompt -> LLM -> JSON Parser
    chain = resume_eval_prompt | llm | parser

    return chain

Overwriting chains/eval_chain.py


In [20]:
import os
from getpass import getpass
from langchain_xai import ChatXAI

print("🔑 Enter your NEW xAI API Key (must start with 'xai-'):")
api_key = getpass()

# 1. Check the format
if not api_key.startswith("xai-"):
    print("❌ WAIT! That key doesn't start with 'xai-'. You might have copied the wrong text.")
else:
    # 2. Save it to memory
    os.environ["XAI_API_KEY"] = api_key
    print("✅ Key format looks perfect. Testing connection to Grok servers...")

    # 3. Ping the server
    try:
        test_llm = ChatXAI(
            model="grok-3-mini",
            temperature=0,
            xai_api_key=api_key
        )
        response = test_llm.invoke("Reply with exactly two words: Connection Successful.")
        print("🎉 SUCCESS! Grok replied:", response.content)
        print("👉 You are cleared for takeoff! Go run your main() cell now.")

    except Exception as e:
        print("❌ Connection failed. Error details:", e)

🔑 Enter your NEW xAI API Key (must start with 'xai-'):
··········
✅ Key format looks perfect. Testing connection to Grok servers...
❌ Connection failed. Error details: Error code: 403 - {'code': 'The caller does not have permission to execute the specified operation', 'error': "Your newly created team doesn't have any credits or licenses yet. You can purchase those on https://console.x.ai/team/5e251301-0bb5-448e-bea6-137104422690."}


In [19]:
# CELL 6
import json
import importlib
import chains.eval_chain

def main():
    # Reload the module to ensure it picks up environment changes
    importlib.reload(chains.eval_chain)
    from chains.eval_chain import get_screening_chain

    # 1. Initialize the LCEL chain
    try:
        screening_chain = get_screening_chain()
    except Exception as e:
        print(f"Initialization Error: {e}")
        return

    # 2. Define the Job Description
    job_description = """
    Data Scientist
    Requirements:
    - 3+ years of experience in Data Science.
    - Strong proficiency in Python, SQL, and Machine Learning algorithms.
    - Experience with LangChain, LLMs, and Generative AI.
    - Tools: Jupyter, Git, Docker.
    """

    # 3. Define the test resumes
    resumes = {
        "Strong Candidate": """
        Data Scientist with 4 years of experience.
        Skills: Machine Learning, Deep Learning, GenAI, Python, SQL.
        Tools: Jupyter, Git, Docker, LangChain, HuggingFace.
        Experience: Built multiple LLM pipelines and predictive models.
        """
    }

    # 4. Process each resume
    for candidate_type, resume_text in resumes.items():
        print(f"--- Evaluating: {candidate_type} ---")
        try:
            result = screening_chain.invoke({
                "job_description": job_description,
                "resume_text": resume_text
            })
            print(json.dumps(result, indent=2))
        except Exception as e:
            print(f"API Error: {e}")
            print("Please ensure your XAI_API_KEY in Cell 2 is correct and starts with 'xai-'.")

if __name__ == "__main__":
    main()

--- Evaluating: Strong Candidate ---
API Error: Error code: 403 - {'code': 'The caller does not have permission to execute the specified operation', 'error': "Your newly created team doesn't have any credits or licenses yet. You can purchase those on https://console.x.ai/team/5e251301-0bb5-448e-bea6-137104422690."}
Please ensure your XAI_API_KEY in Cell 2 is correct and starts with 'xai-'.
